# 第2回講義 宿題

## 課題
今回のLessonで学んだことを元に，MNISTのファッション版 (Fashion MNIST，クラス数10) をソフトマックス回帰によって分類してみましょう．

Fashion MNISTの詳細については以下のリンクを参考にしてください．

Fashion MNIST: https://github.com/zalandoresearch/fashion-mnist

### 目標値
Accuracy: 80%

### ルール
- 訓練データは`x_train`， `y_train`，テストデータは`x_test`で与えられます．
- 予測ラベルは one_hot表現ではなく0~9のクラスラベル で表してください．
- **下のセルで指定されている`x_train、y_train`以外の学習データは使わないでください．**
- **ソフトマックス回帰のアルゴリズム部分の実装はnumpyのみで行ってください** (sklearnやtensorflowなどは使用しないでください)．
    - データの前処理部分でsklearnの関数を使う (例えば `sklearn.model_selection.train_test_split`) のは問題ありません．

### 提出方法
- 2つのファイルを提出していただきます．
    1. テストデータ (`x_test`) に対する予測ラベルを`submission_pred.csv`として保存し，**Omnicampusの宿題タブから「第2回 機械学習基礎」を選択して**提出してください．
    2. それに対応するpythonのコードを`submission_code.py`として保存し，**Omnicampusの宿題タブから「第2回 機械学習基礎 (code)」を選択して**提出してください．pythonファイル自体の提出ではなく，「提出内容」の部分にコードをコピー&ペーストしてください．
      
- なお，採点は1で行い，2はコードの確認用として利用します（成績優秀者はコード内容を公開させていただくかもしれません）．コードの内容を変更した場合は，**1と2の両方を提出し直してください**．

### 評価方法
- 予測ラベルの`y_test`に対する精度 (Accuracy) で評価します．
- 即時採点しLeader Boardを更新します（採点スケジュールは別アナウンス）．
- 締切時の点数を最終的な評価とします．

### ドライブのマウント

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### データの読み込み（このセルは修正しないでください）

In [60]:
import os
import sys

import numpy as np
import pandas as pd

sys.modules['tensorflow'] = None

def load_fashionmnist():
    # 学習データ
    x_train = np.load('drive/MyDrive/DLBasic/HW/HW2/data/x_train.npy')
    y_train = np.load('drive/MyDrive/DLBasic/HW/HW2/data/y_train.npy')

    # テストデータ
    x_test = np.load('drive/MyDrive/DLBasic/HW/HW2/data/x_test.npy')

    x_train = x_train.reshape(-1, 784).astype('float32') / 255
    y_train = np.eye(10)[y_train.astype('int32')]
    x_test = x_test.reshape(-1, 784).astype('float32') / 255

    return x_train, y_train, x_test

### ソフトマックス回帰の実装

In [61]:
# logの中身が0になるのを防ぐ
def np_log(x):
    return np.log(np.clip(a=x, a_min=1e-10, a_max=1e+10))

In [122]:
class AdamOptimizer:
    def __init__(self, shape, lr=0.9, beta1=0.9, beta2=0.999, eps=1e-8):
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.m = np.zeros(shape)
        self.v = np.zeros(shape)
        self.t = 0

    def update(self, param, grad):
        self.t += 1
        self.m = self.beta1 * self.m + (1 - self.beta1) * grad
        self.v = self.beta2 * self.v + (1 - self.beta2) * (grad ** 2)
        m_hat = self.m / (1 - self.beta1 ** self.t)
        v_hat = self.v / (1 - self.beta2 ** self.t)
        return param - self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

In [123]:
import numpy as np
np.random.seed(42)

x_train, y_train, x_test = load_fashionmnist()

from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

def softmax(x, axis=1):
    x -= x.max(axis, keepdims=True)
    x_exp = np.exp(x)
    return x_exp / x_exp.sum(axis, keepdims=True)

# 重み
W = np.random.uniform(low=-0.08, high=0.08, size=(784, 10)).astype('float32')
b = np.zeros(shape=(10,)).astype('float32')

adam_w = AdamOptimizer(W.shape)
adam_b = AdamOptimizer(b.shape)



# 学習データと検証データに分割
x_train, x_valid, y_train, y_valid = train_test_split(x_train, y_train, test_size=0.1)

def train(x, t, update_lr=True, eps=1):
    global W, b

    batch_size = x.shape[0]

    y_hat = softmax(np.matmul(x, W) + b)

    cost = (- t * np_log(y_hat)).sum(axis=1).mean()
    delta = y_hat - t

    dW = np.matmul(x.T, delta) / batch_size
    db = np.matmul(np.ones(shape=(batch_size,)), delta) / batch_size

    # W -= eps * dW
    # b -= eps * db

    if update_lr:
      W = adam_w.update(W, dW)
      b = adam_b.update(b, db)


    return cost

def valid(x, t):
    y_hat = softmax(np.matmul(x, W) + b)
    cost = (-t * np_log(y_hat)).sum(axis=1).mean()

    return cost, y_hat

lr = 1

for epoch in range(100):

    cost = train(x_train, y_train, True, lr)

    cost, y_pred = valid(x_valid, y_valid)

    if epoch % 10 == 9 or epoch == 0:
        accuracy  = accuracy_score(y_valid.argmax(axis=1), y_pred.argmax(axis=1))
        print('EPOCH: {}, Valid Cost: {:.3f}, Valid Accuracy: {:.3f}'.format(
            epoch + 1,
            cost,
            accuracy
        ))

EPOCH: 1, Valid Cost: 13.992, Valid Accuracy: 0.264
EPOCH: 10, Valid Cost: 8.132, Valid Accuracy: 0.615
EPOCH: 20, Valid Cost: 6.275, Valid Accuracy: 0.689
EPOCH: 30, Valid Cost: 4.315, Valid Accuracy: 0.767
EPOCH: 40, Valid Cost: 4.014, Valid Accuracy: 0.776
EPOCH: 50, Valid Cost: 3.120, Valid Accuracy: 0.814
EPOCH: 60, Valid Cost: 2.679, Valid Accuracy: 0.824
EPOCH: 70, Valid Cost: 2.320, Valid Accuracy: 0.831
EPOCH: 80, Valid Cost: 1.953, Valid Accuracy: 0.836
EPOCH: 90, Valid Cost: 4.385, Valid Accuracy: 0.749
EPOCH: 100, Valid Cost: 3.436, Valid Accuracy: 0.768


In [124]:
y_one_hot = softmax(np.matmul(x_test, W) + b)
y_pred = np.argmax(y_one_hot, axis=1)

submission = pd.Series(y_pred, name='label')
submission.to_csv('drive/MyDrive/DLBasic/HW/HW2/submission_pred.csv', header=True, index_label='id')